# 1주차 6교시 — 표 결합·날짜·학습 데이터 분할하기

## 학생용 지적 실험 노트

**수업이 끝나면 할 수 있는 것:** 회원과 선호조건을 key로 결합하고 날짜 파생변수를 만든 뒤 X·y와 train·test를 안전하게 나눕니다.

> 실행 전에 반드시 예상 답을 적습니다. `___` 또는 `FILL_ME`만 채운 뒤 실행하고, 값을 한 번 바꾸고, 결과가 왜 달라졌는지 한 문장으로 적습니다.


## 지금 열 파일과 수업 진행 방법

- 지금 열 파일: `1주차_6교시_학생용_지적실험노트.ipynb`
- 함께 받은 `matchmaking_data/01_week1` 폴더의 CSV를 사용합니다. 폴더 위치는 바꾸지 않습니다.
- 강사가 PPT에서 Q번호를 보여주면 노트북에서 같은 Q번호를 찾습니다.
- 먼저 결과를 예상한 뒤 빈칸을 채우고 실행합니다.
- 교시가 끝나면 자신이 작성한 노트북을 저장합니다.



## 50분 운영 리듬

- 0~5분 · 두 표와 공통 key
- 5~16분 · merge·validate
- 16~25분 · datetime·dt
- 25~31분 · 경과일
- 31~38분 · X·y
- 38~47분 · split·stratify
- 47~50분 · 누수 복구·새 문제 풀기


## 준비 셀 — 데이터와 라이브러리 연결

- **뜻:** 오늘 사용할 도구와 데이터 폴더를 준비합니다.
- **이유:** 뒤의 Q가 데이터 위치와 앞 교시에 의존하지 않게 만듭니다.
- **정상:** 데이터 폴더 경로가 출력됩니다.
- **복구:** 프로젝트 폴더 안에서 노트북을 열고 이 셀부터 다시 실행합니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_week1_data_dir():
    """어디에서 노트북을 열어도 수업 데이터 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base / "course_materials" / "matchmaking_data" / "01_week1",
            base / "matchmaking_data" / "01_week1",
            base / "01_week1",
        ]
        for candidate in candidates:
            if (candidate / "matchmaking_members.csv").exists():
                return candidate
    raise FileNotFoundError("matchmaking_members.csv를 찾지 못했습니다. 프로젝트 폴더 안에서 노트북을 열어 주세요.")

DATA_DIR = find_week1_data_dir()
print("데이터 폴더:", DATA_DIR)
from sklearn.model_selection import train_test_split


## 수업에서 사용할 데이터 준비

이 셀은 미리 제공하는 준비 코드입니다. 학생은 따라 치지 않고 실행 결과의 행·열 수만 확인합니다.


In [ ]:
members = pd.read_csv(DATA_DIR / "matchmaking_members.csv")
preferences = pd.read_csv(DATA_DIR / "matchmaking_preferences.csv")
members = members.drop_duplicates("member_id", keep="first").copy()
members["annual_income_10k_krw"] = pd.to_numeric(members["annual_income_10k_krw"], errors="coerce")
print("회원:", members.shape, "선호:", preferences.shape)


## Q1. 행 순서가 아니라 member_id로 두 표를 연결하세요

> **처음 배우는 사람을 위한 안내**
>
> 뜻: merge는 양쪽 표의 공통 key 값이 같은 행을 찾아 열을 옆으로 붙입니다.
> 
> 이유: 회원 정보와 이상형 조건이 서로 다른 파일에 나뉘어 있기 때문입니다.
> 
> 정상: left merge 결과 1000행, 회원 21열+선호 16개 추가열=37열입니다.
> 
> 복구: on 철자, 양쪽 dtype, key 중복을 확인하고 validate로 관계를 검사합니다.

### 실행 전에 생각할 질문

how='left'는 회원 1000명을 모두 유지한다는 수업 정책입니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
joined = pd.___(
    members,
    preferences,
    on=___,
    how=___,
    validate="one_to_one"
)
print(joined.shape)


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q2. validate 오류는 왜 ‘좋은 실패’일까요?

> **처음 배우는 사람을 위한 안내**
>
> 뜻: validate='one_to_one'은 양쪽 key가 한 번씩만 나오는지 조건으로 검사합니다.
> 
> 이유: 중복 key로 행이 조용히 늘어나는 결합 사고를 즉시 발견하기 위해서입니다.
> 
> 정상: 원본 중복 회원으로 시도하면 MergeError, 중복 제거 뒤 성공합니다.
> 
> 복구: 오류를 끄지 말고 duplicated('member_id') 결과를 조사합니다.

### 실행 전에 생각할 질문

이 셀은 의도적 오류 실험입니다. 오류 이름과 원인만 읽고 다음 셀에서 복구합니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
members_with_duplicates = pd.read_csv(DATA_DIR / "matchmaking_members.csv")
print("중복 추가행:", members_with_duplicates.duplicated("member_id").sum())

# 강사 지시에 따라 아래 결합을 실행하고 오류 마지막 줄을 읽으세요.
# pd.merge(members_with_duplicates, preferences, on="member_id", how="left", validate="one_to_one")


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q3. 문자열 날짜를 계산 가능한 datetime으로 바꾸세요

> **처음 배우는 사람을 위한 안내**
>
> 뜻: to_datetime은 글자 날짜를 시간 자료형으로 바꾸고 errors='coerce'는 잘못된 날짜를 NaT로 표시합니다.
> 
> 이유: 연·월 추출, 정렬, 날짜 차이를 안전하게 하기 위해서입니다.
> 
> 정상: 잘못된 날짜 5개가 NaT, signup_year·signup_month가 생성됩니다.
> 
> 복구: 원본 문자열을 보존한 채 변환 실패 행을 먼저 출력합니다.

### 실행 전에 생각할 질문

errors='coerce'는 오류를 숨기는 옵션이 아니라 조사할 실패 위치를 표시하는 정책입니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
joined["signup_dt"] = pd.___(joined["signup_datetime"], errors=___)
print("잘못된 날짜:", joined["signup_dt"].isna().sum())
joined["signup_year"] = joined["signup_dt"].dt.___
joined["signup_month"] = joined["signup_dt"].dt.___
print(joined[["signup_datetime", "signup_dt", "signup_year", "signup_month"]].head())


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q4. 기준일에서 가입일을 빼 가입 경과일을 만드세요

> **처음 배우는 사람을 위한 안내**
>
> 뜻: Timestamp끼리 빼면 시간 간격이 되고 .dt.days로 일 수를 꺼냅니다.
> 
> 이유: 문자열 가입일 자체보다 ‘얼마나 오래됐는가’가 모델과 상담 질문에 더 직접적이기 때문입니다.
> 
> 정상: signup_days 열은 숫자이며 잘못된 날짜 5행만 NaN입니다.
> 
> 복구: 기준일을 수업에서 고정하고 미래 가입일 음수 여부를 점검합니다.

### 실행 전에 생각할 질문

오늘 날짜를 자동 사용하면 수업 때마다 답이 달라지므로 2026-09-01로 고정합니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
REFERENCE_DATE = pd.Timestamp("2026-09-01")
joined["signup_days"] = (___ - joined["signup_dt"]).dt.___
print(joined[["member_id", "signup_dt", "signup_days"]].head())


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q5. 정답 y와 입력 X를 분리해 답안지 유출을 막으세요

> **처음 배우는 사람을 위한 안내**
>
> 뜻: y는 예측할 meeting_scheduled, X는 예측 시점에 사용할 입력 특성입니다.
> 
> 이유: 정답을 X에 포함하면 모델이 답안지를 본 것 같은 가짜 성능을 냅니다.
> 
> 정상: y 길이 1000, X에 meeting_scheduled가 없습니다.
> 
> 복구: X.columns에서 목표 열과 개인식별 열을 다시 확인합니다.

### 실행 전에 생각할 질문

member_id와 name도 예측 입력에서 빼는 이유는 식별자 기억과 개인정보 최소화입니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
drop_columns = ["meeting_scheduled", "member_id", "name", "signup_datetime", "signup_dt"]
X = joined.___(columns=drop_columns)
y = joined[___]
print(X.shape, y.shape)
print("정답 열이 X에 있나?", "meeting_scheduled" in X.columns)


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q6. train/test를 나누고 stratify가 비율을 지켰는지 확인하세요

> **처음 배우는 사람을 위한 안내**
>
> 뜻: train은 모델이 패턴을 배우는 자료이고, test는 처음 보는 자료에서 잘 작동하는지 확인하는 자료입니다. random_state는 같은 분할, stratify는 정답 비율을 보존합니다.
> 
> 이유: 같은 자료로 배우고 시험하는 과대평가와 소수 범주의 쏠림을 막습니다.
> 
> 정상: 800행/200행이며 True 비율이 전체·train·test에서 비슷합니다.
> 
> 복구: X와 y 길이, 네 반환값 순서, stratify=y를 확인합니다.

### 실행 전에 생각할 질문

실행 전 test 행 수와 True 비율이 약 42.8%임을 예상하세요.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=___,
    random_state=___,
    stratify=___
)
print(X_train.shape, X_test.shape)
print("전체", y.value_counts(normalize=True).sort_index())
print("train", y_train.value_counts(normalize=True).sort_index())
print("test", y_test.value_counts(normalize=True).sort_index())


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q7. 빈칸을 채워 다른 분할에서도 조건이 유지되는지 확인하기

> **처음 배우는 사람을 위한 안내**
>
> 성공 조건: random_state=7로 다시 분할하고 800/200, 열 동일, y 비율 차이 0.01 미만을 assert합니다.
> 
> 빈칸 안내: test 비율, 새로운 random_state와 비율을 유지할 기준만 채웁니다. 아래 assert 코드는 결과를 자동으로 검사하는 제공 코드입니다.
> 
> 정상: 모든 assert 통과, 7교시에는 X_train에만 fit합니다.
> 
> 복구: 분할 변수 이름을 섞지 말고 _v2 접미사를 붙입니다.

### 실행 전에 생각할 질문

분할에 필요한 세 빈칸을 채운 뒤, 제공된 검사 코드가 모두 통과하는지 확인하세요.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X, y,
    test_size=___,
    random_state=___,
    stratify=___
)

# 아래는 결과가 조건에 맞는지 확인하는 제공 코드입니다.
assert len(X_train_v2) == 800
assert len(X_test_v2) == 200
assert X_train_v2.columns.tolist() == X_test_v2.columns.tolist()
assert abs(y_train_v2.mean() - y_test_v2.mean()) < 0.01

print("모든 분할 조건을 통과했습니다.")


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## 마무리 질문

1. 오늘 가장 자주 쓸 기능 하나와 그 이유는?
2. 오늘 기능을 쓰게 되는 데이터의 신호는?
3. 정상 결과를 무엇으로 확인했나?
4. 다음 교시로 넘겨야 할 결과는?
